# 第06章 C组人工复核

## 6.1 PINN：原理与实现

（1）日期：2026-11-15；组别：G13（C2组）。

（2）审核材料：`A2-G15-正文 (1)(1).md` 第6.1节；`B2组-修正清单(1).md` 的CH06-G09-B011、CH06-G09-B009。

（3）复核边界：本Notebook是C组独立数学与数值检查，不是A2的原始实验Notebook，也没有重跑A2的1500步Adam与L-BFGS PINN训练。A2原始Notebook未上传，故B2报告的原单元代码与执行计数未能由C组独立复测。

（4）运行环境：Python 3、NumPy；CPU即可。下列代码单元按顺序执行，输出由本Notebook实际运行生成。

### 6.1-V1 解析解与边界条件

对
\[
-u''(x)=\pi^2\sin(\pi x),\quad u(0)=u(1)=0
\]
取 \(u_*(x)=\sin(\pi x)\)，解析上有 \(u_*''=-\pi^2\sin(\pi x)\)。本单元同时检查解析残差、边界值，并用中心差分近似二阶导数作数值交叉检查。

In [1]:
import numpy as np
import platform

print("Python:", platform.python_version(), "| NumPy:", np.__version__)

def u_exact(x):
    x = np.asarray(x, dtype=float)
    return np.sin(np.pi * x)

def source(x):
    x = np.asarray(x, dtype=float)
    return np.pi**2 * np.sin(np.pi * x)

def u_exact_xx(x):
    x = np.asarray(x, dtype=float)
    return -np.pi**2 * np.sin(np.pi * x)

x = np.linspace(0.01, 0.99, 201)
r_analytic = -u_exact_xx(x) - source(x)
h = 1e-4
u_xx_fd = (u_exact(x+h) - 2*u_exact(x) + u_exact(x-h)) / h**2
r_fd = -u_xx_fd - source(x)
boundary_max = np.max(np.abs(u_exact(np.array([0.0, 1.0]))))

print("V1 boundary max =", f"{boundary_max:.3e}")
print("V1 analytic residual max =", f"{np.max(np.abs(r_analytic)):.3e}")
print("V1 central-difference residual max =", f"{np.max(np.abs(r_fd)):.3e}")

assert boundary_max < 1e-12
assert np.max(np.abs(r_analytic)) < 1e-12
assert np.max(np.abs(r_fd)) < 1e-6

Python: 3.13.5 | NumPy: 2.3.5
V1 boundary max = 1.225e-16
V1 analytic residual max = 0.000e+00
V1 central-difference residual max = 9.705e-08


### 6.1-V2 反例：残差为零但边界不满足

检查
\[
v(x)=\sin(\pi x)+3x+2.
\]
其解析二阶导数与真解相同，因此内部残差为零；但端点值分别为2和5。这里直接从实际函数导数构造残差，并另用中心差分检查，避免“先把残差变量设成0，再断言它等于0”的空洞自检。

In [2]:
def v(x):
    x = np.asarray(x, dtype=float)
    return np.sin(np.pi*x) + 3.0*x + 2.0

def v_xx(x):
    x = np.asarray(x, dtype=float)
    return -np.pi**2*np.sin(np.pi*x)

r_v_analytic = -v_xx(x) - source(x)
v_xx_fd = (v(x+h) - 2*v(x) + v(x-h)) / h**2
r_v_fd = -v_xx_fd - source(x)

ends = v(np.array([0.0, 1.0]))
L_b = np.mean(ends**2)
L_f = np.mean(r_v_analytic**2)
lambda_b = 10.0
L_total = L_f + lambda_b*L_b

print("V2 endpoint values =", ends.tolist())
print("V2 analytic residual max =", f"{np.max(np.abs(r_v_analytic)):.3e}")
print("V2 finite-difference residual max =", f"{np.max(np.abs(r_v_fd)):.3e}")
print("V2 boundary loss =", f"{L_b:.6f}")
print("V2 total loss for lambda_b=10 =", f"{L_total:.6f}")

assert np.allclose(ends, [2.0, 5.0], atol=1e-12)
assert np.max(np.abs(r_v_analytic)) < 1e-12
assert np.max(np.abs(r_v_fd)) < 1e-6
assert np.isclose(L_b, 14.5)
assert np.isclose(L_total, 145.0)

V2 endpoint values = [2.0, 5.0]
V2 analytic residual max = 0.000e+00
V2 finite-difference residual max = 2.801e-07
V2 boundary loss = 14.500000
V2 total loss for lambda_b=10 = 145.000000


### 6.1-V3 内部方程的仿射自由度

若 \(u_*(x)\) 是内部方程的一个解，则 \(u_*(x)+ax+b\) 仍满足相同内部方程，因为一次函数的二阶导数为0。两个零边界分别给出 \(b=0\) 与 \(a+b=0\)，从而唯一确定 \(a=b=0\)。

In [3]:
affine_cases = [(0.0, 0.0), (3.0, 2.0), (-2.0, 0.5)]
for a, b in affine_cases:
    # -d2/dx2 [sin(pi*x) + a*x + b] - source(x) = 0
    residual = -u_exact_xx(x) - source(x)
    endpoint_values = np.array([b, a+b])
    print(f"a={a:5.1f}, b={b:5.1f}, residual_max={np.max(np.abs(residual)):.1e}, "
          f"endpoints={endpoint_values.tolist()}")

# If both endpoints are zero, solve b=0 and a+b=0.
b_from_left = 0.0
a_from_right = -b_from_left
assert a_from_right == 0.0 and b_from_left == 0.0
print("V3 zero-boundary constraints imply a=0, b=0.")

a=  0.0, b=  0.0, residual_max=0.0e+00, endpoints=[0.0, 0.0]
a=  3.0, b=  2.0, residual_max=0.0e+00, endpoints=[2.0, 5.0]
a= -2.0, b=  0.5, residual_max=0.0e+00, endpoints=[0.5, -1.5]
V3 zero-boundary constraints imply a=0, b=0.


### 6.1-V4 硬边界构造的二阶导数

令 \(u(x)=x(1-x)N(x)\)。完整乘积求导为
\[
u''=-2N+2(1-2x)N'+x(1-x)N''.
\]
本单元用二次多项式 \(N(x)\) 比较该解析表达式与中心差分结果，并检查两个端点严格为0。

In [4]:
def N(x):
    return 1.0 + 2.0*x + 0.5*x**2

def N_x(x):
    return 2.0 + x

def N_xx(x):
    return np.ones_like(np.asarray(x, dtype=float))

def u_hard(x):
    return x*(1.0-x)*N(x)

x_hard = np.linspace(0.05, 0.95, 101)
u_xx_formula = (-2.0*N(x_hard)
                + 2.0*(1.0-2.0*x_hard)*N_x(x_hard)
                + x_hard*(1.0-x_hard)*N_xx(x_hard))
h_hard = 1e-4
u_xx_fd_hard = (u_hard(x_hard+h_hard)-2*u_hard(x_hard)
                +u_hard(x_hard-h_hard))/h_hard**2
hard_boundary = u_hard(np.array([0.0, 1.0]))

print("V4 hard boundary values =", hard_boundary.tolist())
print("V4 second-derivative max difference =",
      f"{np.max(np.abs(u_xx_formula-u_xx_fd_hard)):.3e}")

assert np.max(np.abs(hard_boundary)) < 1e-12
assert np.max(np.abs(u_xx_formula-u_xx_fd_hard)) < 1e-6

V4 hard boundary values = [0.0, 0.0]
V4 second-derivative max difference = 4.311e-08


### 6.1-V5 有限差分基线与二阶收敛

使用内部节点 \(x_i=ih\)、\(h=1/(n+1)\)，通过Thomas三对角算法求解
\[
(-U_{i-1}+2U_i-U_{i+1})/h^2=\pi^2\sin(\pi x_i).
\]
与解析解在同一组节点上比较最大误差，并按实际网格比计算观测阶。

In [5]:
def fd_thomas(n):
    h = 1.0/(n+1)
    nodes = np.arange(1, n+1)*h
    rhs = source(nodes).copy()
    diag = np.full(n, 2.0/h**2)
    off = -1.0/h**2

    # Forward elimination for the tridiagonal system.
    for i in range(1, n):
        factor = off/diag[i-1]
        diag[i] -= factor*off
        rhs[i] -= factor*rhs[i-1]

    # Back substitution.
    U = np.empty(n)
    U[-1] = rhs[-1]/diag[-1]
    for i in range(n-2, -1, -1):
        U[i] = (rhs[i]-off*U[i+1])/diag[i]
    return nodes, U

ns = [15, 31, 63, 127]
hs, errors = [], []
for n in ns:
    nodes, U = fd_thomas(n)
    err = np.max(np.abs(U-u_exact(nodes)))
    h_grid = 1.0/(n+1)
    hs.append(h_grid)
    errors.append(err)
    p = None if len(errors) == 1 else np.log(errors[-2]/errors[-1])/np.log(hs[-2]/hs[-1])
    print(f"n={n:3d}, h={h_grid:.8f}, max_error={err:.9e}, "
          f"observed_order={'—' if p is None else f'{p:.4f}'}")

# For odd n the midpoint is a node, and the exact discrete amplitude is known.
for n, err in zip(ns, errors):
    h_grid = 1.0/(n+1)
    C_h = np.pi**2*h_grid**2/(2.0-2.0*np.cos(np.pi*h_grid))
    exact_max_error = abs(C_h-1.0)
    assert np.isclose(err, exact_max_error, rtol=1e-9, atol=1e-12)

orders = [np.log(errors[i-1]/errors[i])/np.log(hs[i-1]/hs[i])
          for i in range(1, len(errors))]
assert all(1.99 < p < 2.02 for p in orders)
print("V5 observed orders =", [round(float(p), 6) for p in orders])

n= 15, h=0.06250000, max_error=3.218964440e-03, observed_order=—
n= 31, h=0.03125000, max_error=8.035776794e-04, observed_order=2.0021
n= 63, h=0.01562500, max_error=2.008218097e-04, observed_order=2.0005
n=127, h=0.00781250, max_error=5.020091591e-05, observed_order=2.0001
V5 observed orders = [2.002087, 2.000522, 2.00013]


### 6.1-V6 独立评价指标不能互相替代

构造仅用于指标演示的函数
\[
u_{\mathrm{test}}(x)=u_*(x)+\epsilon x(1-x),\quad \epsilon=10^{-3}.
\]
该扰动在两个端点为0，但其二阶导数改变了内部残差。由此分别计算相对解误差、最大解误差、边界误差和残差RMS。它不是A2训练出来的网络输出。

In [6]:
grid = np.linspace(0.0, 1.0, 1001)
eps = 1e-3
u_true = u_exact(grid)
u_test = u_true + eps*grid*(1.0-grid)
err = u_test-u_true

E_rel = np.linalg.norm(err)/np.linalg.norm(u_true)
E_inf = np.max(np.abs(err))
E_b = np.max(np.abs(u_test[[0, -1]]))

# Since d2/dx2 [eps*x*(1-x)] = -2*eps, residual is exactly 2*eps.
r_test = np.full(grid[1:-1].shape, 2.0*eps)
R_rms = np.sqrt(np.mean(r_test**2))

print("V6 relative L2 error =", f"{E_rel:.6e}")
print("V6 maximum solution error =", f"{E_inf:.6e}")
print("V6 boundary maximum error =", f"{E_b:.6e}")
print("V6 interior residual RMS =", f"{R_rms:.6e}")

assert E_rel > 0 and E_inf > 0
assert E_b < 1e-12
assert np.isclose(R_rms, 2.0*eps)

V6 relative L2 error = 2.581989e-04
V6 maximum solution error = 2.500000e-04
V6 boundary maximum error = 1.224647e-16
V6 interior residual RMS = 2.000000e-03


### 6.1-V7 对B2报告事项的证据边界

B2 CH06-G09-B011报告：A2原Notebook的“解析反例”自检只检查 `residual_max == 0`，但没有把反例函数代入求导算子。B2 CH06-G09-B009报告：A2整份Notebook的执行计数存在分段运行痕迹。

本轮没有上传A2原始实验Notebook，因此C组不能直接查看这些单元源码或执行计数。V2是C组独立编写的正确残差检查，不等于复测了A2原单元。B2还在其“未发现问题”部分称A2 6.1、6.2正文数字可在Notebook保存输出中找到；这仍属于B2的静态核对结论，不是C组独立运行结果。

### 复核结果与未覆盖范围

| 编号 | 独立检查 | 判断 |
| --- | --- | --- |
| V1 | 解析解边界与方程残差 | 已运行，通过 |
| V2 | 反例函数实际残差、端点值与边界损失 | 已运行，通过 |
| V3 | 内部方程仿射自由度与边界唯一性 | 已运行，通过 |
| V4 | 硬边界构造的完整二阶导数 | 已运行，通过 |
| V5 | Thomas有限差分误差与二阶收敛 | 已运行，通过 |
| V6 | 解误差、边界误差和残差RMS的区别 | 已运行，通过 |

未覆盖：A2的PyTorch PINN完整训练、三种随机种子重跑、A2原Notebook保存输出和执行计数核对、B2 B011原单元源码复测、A1/B1版本比较。以上事项均不得写成C组已独立复现。